# pinn_ndof_chain_sim_tf2_free_free_icv.ipynb

Comprehensive PINN workflow for a **20-DOF free-free chain** with:
- **no external force**,
- **left-end initial velocity excitation only**,
- sequential simulation up to **50 impacts**.

This notebook is designed to mirror the structure of `pinn_ndof_chain_sim_tf2_freq2_A10.ipynb`, but for the free-free/no-force scenario.


In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from pinn_ndof_chain_tf2_free_free_no_force import (
    PIPNNs,
    build_free_free_chain_matrices,
    make_left_velocity_ic,
    find_impact_times,
    propagate_ics,
    newmark_beta,
)

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
np.random.seed(1234)
tf.random.set_seed(1234)
print('TF version:', tf.__version__)


## 1) Parameters


In [ ]:
# --- physical system ---
n_dof = 20
m_x = 1.0
m_y = 0.3
k = 1.0
c = 0.0
D = 1.0
r = 1.0

# no external forcing in this study
phi1 = 0.0
phi2 = 0.0

# left-end initial excitation: x1(0)=x1_0, x1_dot(0)=v1_0
x1_0 = 0.0
v1_0 = 1.0

# --- PINN/training controls ---
layers = [1, 64, n_dof]
hyp_ini_weight_loss = np.array([1.0, 1.0])
optimizer_LB_value = True

# per-segment horizon for impact search
T_segment = 1.0
n_t_segment = 1000
nIter_per_segment = 2000

# target number of impacts to simulate
n_impacts_target = 50

# optional Newmark reference for early-time sanity check
T_ref = 8.0
dt_ref = 0.001


## 2) Build free-free matrices and initial conditions


In [ ]:
M_total, C_total, K_total = build_free_free_chain_matrices(
    n_dof=n_dof, m_x=m_x, k=k, c=c
)

x0_total, xt0_total = make_left_velocity_ic(
    n_dof=n_dof, x1_0=x1_0, v1_0=v1_0
)

y0 = np.zeros(n_dof)
yt0 = np.zeros(n_dof)

# impact mapping matrix (same collision law structure)
A = np.array([[m_x, m_y],
              [1.0, -1.0]])
B = np.array([[m_x, m_y],
              [-r,   r  ]])
A_inv_B = np.linalg.inv(A) @ B

print('M/K/C shape:', M_total.shape, K_total.shape, C_total.shape)
print('Initial x1, v1:', x0_total[0, 0], xt0_total[0, 0])


## 3) Newmark early-time reference (no impactors, no external force)


In [ ]:
num_ref = int(T_ref / dt_ref) + 1
t_ref = np.linspace(0, T_ref, num_ref)
F_ref = np.zeros((n_dof, num_ref))

u_ref, v_ref, _ = newmark_beta(
    M_total, C_total, K_total, F_ref,
    dt=dt_ref, n_steps=num_ref, n_dof=n_dof,
    x0=x0_total[0], xt0=xt0_total[0]
)

plt.figure(figsize=(10, 3))
for i in [0, 4, 9, 14, 19]:
    plt.plot(t_ref, u_ref[i, :], lw=1.2, label=f'DOF {i+1}')
plt.title('Free-free no-force Newmark reference (selected DOFs)')
plt.xlabel('Time (s)')
plt.ylabel('Displacement')
plt.legend(ncol=5, fontsize=8)
plt.tight_layout()
plt.show()


## 4) Multi-impact PINN driver (up to 50 impacts)


In [ ]:
def run_free_free_pinn_multiimpact(
    n_impacts_target,
    T_segment,
    n_t_segment,
    nIter_per_segment,
):
    lb = np.array([0.0])
    ub = np.array([float(T_segment)])

    x0_cur = x0_total.copy()
    xt0_cur = xt0_total.copy()
    y0_cur = y0.copy()
    yt0_cur = yt0.copy()

    phi_cur = np.array([[0.0]])
    t_global = 0.0

    t_hist, x_hist, xt_hist, y_hist = [], [], [], []
    impact_log = []
    n_impacts_found = 0

    # keep advancing by fixed 1 s windows until 50 impacts are collected
    seg_id = 0
    while n_impacts_found < n_impacts_target:
        seg_id += 1
        t_seg = np.linspace(0.0, float(T_segment), int(n_t_segment)).reshape(-1, 1)

        model = PIPNNs(
            lb, ub,
            np.array([[0.0]]), t_seg,
            x0_cur, xt0_cur,
            y0_cur, yt0_cur,
            M_total, K_total, D, n_dof,
            phi_cur, phi1, phi2,
            layers,
            hyp_ini_weight_loss,
            C=C_total,
            optimizer_LB=optimizer_LB_value,
        )
        model.train(nIter=int(nIter_per_segment), optimizer_LB=optimizer_LB_value)

        t_impacts, hit = find_impact_times(model, y0_cur, yt0_cur, D, float(T_segment))

        if np.any(hit):
            # impact exists inside this window: stop at earliest hit and apply velocity jump
            j = int(np.argmin(t_impacts))
            t_event = float(t_impacts[j])
            has_impact = np.isfinite(t_event) and (t_event > 0.0) and (t_event < float(T_segment))
        else:
            # no impacts in this 1 s window: march full window with NO jump
            j = None
            t_event = float(T_segment)
            has_impact = False

        if not np.isfinite(t_event) or (t_event <= 0.0):
            print(f'[stop] invalid segment event time at segment {seg_id}: {t_event}')
            break

        # local trajectories up to event time (impact time or end of window)
        t_local = np.linspace(0.0, t_event, int(n_t_segment) + 1).reshape(-1, 1)
        x_local, xt_local, _ = model.predict(t_local)
        y_local = (y0_cur + np.outer(t_local.flatten(), yt0_cur))

        # stitch to global history (avoid duplicated point at segment joints)
        t_global_local = t_global + t_local.flatten()
        if len(t_hist) > 0:
            t_global_local = t_global_local[1:]
            x_local = x_local[1:, :]
            xt_local = xt_local[1:, :]
            y_local = y_local[1:, :]

        t_hist.append(t_global_local)
        x_hist.append(x_local)
        xt_hist.append(xt_local)
        y_hist.append(y_local)

        # state at event for IC propagation
        x_hit, xt_hit, _ = model.predict(np.array([[t_event]], dtype=np.float32))

        if has_impact:
            x0_next, xt0_next, y0_next, yt0_next = propagate_ics(
                model,
                t_event,
                x_hit, xt_hit,
                y0_cur, yt0_cur,
                j,
                m_x * np.ones(n_dof), m_y * np.ones(n_dof), r,
                A_inv_B,
            )

            n_impacts_found += 1
            impact_log.append({
                'impact_id': n_impacts_found,
                'segment_id': seg_id,
                'dof': j + 1,
                't_local': t_event,
                't_global': t_global + t_event,
            })
            print(
                f"segment {seg_id:03d}: impact {n_impacts_found:02d}/{n_impacts_target} "
                f"at DOF {j+1:02d}, t_local={t_event:.6f}, t_global={t_global + t_event:.6f}"
            )
        else:
            # no-impact segment: carry end-of-window state forward unchanged in velocity law
            x0_next = x_hit.copy()
            xt0_next = xt_hit.copy()
            y0_next = y0_cur + yt0_cur * t_event
            yt0_next = yt0_cur.copy()
            print(
                f"segment {seg_id:03d}: no impact in [0, {T_segment:.2f}] s, "
                f"advance without velocity jump."
            )

        t_global += t_event
        phi_cur = phi_cur + t_event
        x0_cur, xt0_cur, y0_cur, yt0_cur = x0_next, xt0_next, y0_next, yt0_next

    if len(t_hist) == 0:
        return None

    out = {
        't': np.concatenate(t_hist),
        'x': np.vstack(x_hist),
        'xt': np.vstack(xt_hist),
        'y': np.vstack(y_hist),
        'impacts': impact_log,
        'n_segments': seg_id,
    }
    return out


## 5) Run simulation for 50 impacts


In [ ]:
results = run_free_free_pinn_multiimpact(
    n_impacts_target=n_impacts_target,
    T_segment=T_segment,
    n_t_segment=n_t_segment,
    nIter_per_segment=nIter_per_segment,
)
if results is None:
    raise RuntimeError('No impacts were simulated. Increase T_segment and/or iterations.')
print('Total impacts simulated:', len(results['impacts']))
print('Total simulated time   :', results['t'][-1])
print('Total segments learned :', results['n_segments'])


## 6) Plot responses and impact timeline


In [ ]:
t = results['t']
x = results['x']
y = results['y']
impacts = results['impacts']

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

axes[0].plot(t, x[:, 0], lw=1.4, label='x1 (left end)')
axes[0].plot(t, x[:, -1], lw=1.2, label=f'x{n_dof} (right end)')
axes[0].set_ylabel('Displacement')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(t, x[:, 0] - y[:, 0], lw=1.2, label='x1 - y1')
axes[1].axhline(D, color='r', ls='--', lw=1.0)
axes[1].axhline(-D, color='r', ls='--', lw=1.0)
axes[1].set_ylabel('Relative disp.')
axes[1].legend()
axes[1].grid(alpha=0.25)

if len(impacts) > 0:
    t_imp_global = np.array([it['t_global'] for it in impacts])
    dof_imp = np.array([it['dof'] for it in impacts])
    axes[2].scatter(t_imp_global, dof_imp, s=25)
axes[2].set_ylabel('Impact DOF id')
axes[2].set_xlabel('Time (s)')
axes[2].set_title('Impact timeline')
axes[2].grid(alpha=0.25)

plt.tight_layout()
plt.show()


## 7) Save results (optional)


In [ ]:
save_name = 'pinn_free_free_20dof_50impacts.npz'
np.savez(
    save_name,
    t=results['t'],
    x=results['x'],
    xt=results['xt'],
    y=results['y'],
    impacts=np.array(results['impacts'], dtype=object),
)
print('Saved:', save_name)
